In [1]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
spark.sql("""
CREATE TABLE IF NOT EXISTS silver.products (
    product_id STRING,
    name STRING,
    category STRING,
    brand STRING,
    price DOUBLE,
    stock_quantity INT,
    rating DOUBLE,
    is_active BOOLEAN,
    price_category STRING,
    stock_status STRING,
    last_updated TIMESTAMP
)
USING DELTA
""")

StatementMeta(, 320a4991-74e0-4faa-8d20-1f3cdb562723, 3, Finished, Available, Finished, False)

DataFrame[]

In [2]:
last_processed_df = spark.sql("SELECT MAX(last_updated) as last_processed FROM silver.products")
last_processed_timestamp = last_processed_df.collect()[0]['last_processed']

if last_processed_timestamp is None:
    last_processed_timestamp = "1900-01-01T00:00:00.000+00:00"

StatementMeta(, 320a4991-74e0-4faa-8d20-1f3cdb562723, 4, Finished, Available, Finished, False)

In [5]:
spark.sql(f"""
CREATE OR REPLACE TEMPORARY VIEW bronze_incremental_products AS
SELECT *
FROM enterpriseRetail.retailLakehouse.bronze.products WHERE ingestion_timestamp > '{last_processed_timestamp}'
""")

StatementMeta(, 320a4991-74e0-4faa-8d20-1f3cdb562723, 7, Finished, Available, Finished, False)

DataFrame[]

In [6]:
spark.sql("""
CREATE OR REPLACE TEMPORARY VIEW silver_incremental_products AS
SELECT
    product_id,
    name,
    category,
    brand,
    CASE
        WHEN price < 0 THEN 0
        ELSE price
    END AS price,
    CASE
        WHEN stock_quantity < 0 THEN 0
        ELSE stock_quantity
    END AS stock_quantity,
    CASE
        WHEN rating < 0 THEN 0
        WHEN rating > 5 THEN 5
        ELSE rating
    END AS rating,
    is_active,
    CASE
        WHEN price > 1000 THEN 'Premium'
        WHEN price > 100 THEN 'Standard'
        ELSE 'Budget'
    END AS price_category,
    CASE
        WHEN stock_quantity = 0 THEN 'Out of Stock'
        WHEN stock_quantity < 10 THEN 'Low Stock'
        WHEN stock_quantity < 50 THEN 'Moderate Stock'
        ELSE 'Sufficient Stock'
    END AS stock_status,
    CURRENT_TIMESTAMP() AS last_updated
FROM bronze_incremental_products
WHERE name IS NOT NULL AND category IS NOT NULL
""")

StatementMeta(, 320a4991-74e0-4faa-8d20-1f3cdb562723, 8, Finished, Available, Finished, False)

DataFrame[]

In [7]:
spark.sql("""
MERGE INTO silver.products target
USING silver_incremental_products source
ON target.product_id = source.product_id
WHEN MATCHED THEN
    UPDATE SET *
WHEN NOT MATCHED THEN
    INSERT *
""")

StatementMeta(, 320a4991-74e0-4faa-8d20-1f3cdb562723, 9, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [8]:
spark.sql("SELECT * FROM silver.products LIMIT 10").show()

StatementMeta(, 320a4991-74e0-4faa-8d20-1f3cdb562723, 10, Finished, Available, Finished, False)

+----------+-----------+-----------+------------+------+--------------+------+---------+--------------+----------------+--------------------+
|product_id|       name|   category|       brand| price|stock_quantity|rating|is_active|price_category|    stock_status|        last_updated|
+----------+-----------+-----------+------------+------+--------------+------+---------+--------------+----------------+--------------------+
|        20| Product 20|      Books|  BeautyGlow|515.79|           726|   1.6|    false|      Standard|Sufficient Stock|2026-03-04 05:19:...|
|        32| Product 32|Electronics|    SportMax|408.99|           801|   1.6|    false|      Standard|Sufficient Stock|2026-03-04 05:19:...|
|       109|Product 109|       Toys|    BookWorm|621.25|           455|   1.6|    false|      Standard|Sufficient Stock|2026-03-04 05:19:...|
|       148|Product 148|       Home|GardenMaster|545.57|           627|   1.6|    false|      Standard|Sufficient Stock|2026-03-04 05:19:...|
|     